In [1]:
from LoadData import load_tetris_dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import numpy as np

In [2]:
# Load the dataset
train_data = load_tetris_dataset("tetris_dataset.bin")
print("Loaded", len(train_data), "entries.")

Loaded 24871 entries.


In [3]:
# Convert the data to tensors
grids = torch.tensor(np.array([entry["grid"] for entry in train_data]))
active_grids = torch.tensor(np.array([entry["active_grid"] for entry in train_data]))
#queue_ints = torch.tensor([entry["queue"] for entry in train_data])
instructions = torch.tensor(np.array([entry["instruction"] for entry in train_data]))

In [4]:
# Pad the queue
max_queue_length = 8
queue_ints = torch.zeros((len(train_data), max_queue_length), dtype=torch.int64)
for i, entry in enumerate(train_data):
    queue = entry["queue"]
    if queue is not None:
        queue_ints[i, :len(queue)] = torch.tensor(queue)

queue_ints = torch.tensor(queue_ints, dtype=torch.int64)
#Convert queue_ints to OneHot Encoding
queue_onehot = torch.nn.functional.one_hot(queue_ints, num_classes=8)

C:\Users\20neu\AppData\Local\Temp\ipykernel_123540\509892194.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  queue_ints = torch.tensor(queue_ints, dtype=torch.int64)


In [5]:
# Create the dataset
class TetrisDataset(torch.utils.data.Dataset):
    def __init__(self, grids, active_grids, queue_onehot, instructions):
        self.grids = grids
        self.active_grids = active_grids
        self.queue_onehot = queue_onehot
        self.instructions = instructions

    def __len__(self):
        return len(self.grids)

    def __getitem__(self, index):
        grid = self.grids[index]
        active_grid = self.active_grids[index]
        queue_onehot = self.queue_onehot[index]
        instruction = self.instructions[index]

        return {
            "grid": grid,
            "active_grid": active_grid,
            "queue_onehot": queue_onehot,
            "instruction": instruction
        }

In [6]:
# Split the dataset
train_size = int(0.8 * len(grids))  # 80% of the data for training
val_size = len(grids) - train_size  # 20% of the data for validation

dataset = TetrisDataset(grids[:train_size], active_grids[:train_size], queue_onehot[:train_size], instructions[:train_size])
validation_dataset = TetrisDataset(grids[train_size:], active_grids[train_size:], queue_onehot[train_size:], instructions[train_size:])

dataloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)
validation_dataloader = torch.utils.data.DataLoader(validation_dataset, shuffle=False)

In [7]:
print(len(dataset))

19896


In [8]:
# Define the model
class TetrisNet(nn.Module):
    def __init__(self, queue_length=8):
        super(TetrisNet, self).__init__()

        # --------------------
        # Grid Branch (CNN)
        # --------------------
        # Input: two channels (grid and active_grid), each 10x20.
        # Convolutional layers extract spatial features.
        self.conv1 = nn.Conv2d(in_channels=2, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # halves spatial dimensions.
        self.vertical_pool = nn.MaxPool2d(1, 2)  # halves spatial dimensions vertically.

        # Apply batch normalization after each convolutional layer.
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(32)
        self.bn3 = nn.BatchNorm2d(64)

        # After three pooling layers: from (10,20) -> (10/2,20/2) = (5,10) after first pool,
        # and then (5/2,10/2) ≈ (2,5) after second pool (using floor division).
        # Thus, output of conv2: (batch, 64, 2, 5) --> flatten size: 64 * 2 * 5 = 640.
        self.fc_grid = nn.Linear(640, 128)
        # --------------------  
        # Queue Branch (Fully Connected)
        # --------------------
        # Input: one-hot encoded queue, assumed shape (queue_length, 8) per sample,
        # which is flattened to a vector of size (queue_length * 8).
        self.fc_queue = nn.Sequential(
            nn.Linear(queue_length * 8, 32),
            nn.ReLU()
        )
        # --------------------
        # Combined Features
        # --------------------
        # Merge the grid branch (128 features) and queue branch (32 features).
        self.fc_combined = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(64, 7)  # Output: 7 neurons, one per move.
        )

    def forward(self, grid, active_grid, queue_onehot):
        """
        Args:
            grid: Tensor of shape (batch_size, 10, 20) representing the static grid.
            active_grid: Tensor of shape (batch_size, 10, 20) representing the active piece.
            queue_onehot: Tensor of shape (batch_size, queue_length * 7) representing the one-hot encoded queue.
        Returns:
            out: Tensor of shape (batch_size, 7) representing logits for the 7 moves.
        """
        # Combine grid and active_grid along the channel dimension to create a tensor of shape (batch, 2, 10, 20)
        x_grid = torch.stack([grid, active_grid], dim=1)
        
        # Pass through CNN branch.
        x = F.relu(self.bn1(self.conv1(x_grid)))        # (batch, 16, 10, 20)
        x = F.relu(self.bn2(self.conv2(x)))         # (batch, 32, 10, 20)
        x = self.vertical_pool(x)                  # (batch, 32, 10, 10)
        x = F.relu(self.bn3(self.conv3(x)))         # (batch, 64, 10, 10)
        x = self.pool(x)                  # (batch, 64, 5, 5)
        x = x.view(x.size(0), -1)         # Flatten to (batch, 1600) 
        x = F.relu(self.fc_grid(x))       # (batch, 128)        
        
        # Process the queue branch.
        x_queue = self.fc_queue(queue_onehot)  # (batch, 32)
        
        # Concatenate the features from both branches.
        x_combined = torch.cat((x, x_queue), dim=1)  # (batch, 128+32 = 160)
        out = self.fc_combined(x_combined)           # (batch, 7)
        return out
#model = TetrisNet(queue_length=8)
#outputs = model(grid, active_grid, queue_onehot.view(queue_onehot.size(0), -1))

In [9]:
# Reset the model
model = TetrisNet(queue_length=8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

TetrisNet(
  (conv1): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (vertical_pool): MaxPool2d(kernel_size=1, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc_grid): Linear(in_features=640, out_features=128, bias=True)
  (fc_queue): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
  )
  (fc_combined): Sequential(
    (0): Dropout(p=0.6, inplace=False)
    (1): Linear(in_features=160, out_features=64, bias=True)
    (2): ReL

In [10]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64*3, shuffle=True)

Training

In [11]:
# Train the model
print(device)
model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-3    )
criterion = nn.CrossEntropyLoss()
for epoch in range(20):
    running_loss = 0.0
    running_val_loss = 0.0
    for batch in dataloader:
        optimizer.zero_grad()
        #Convert data to floats
        grid = batch["grid"].float().to(device)
        active_grid = batch["active_grid"].float().to(device)
        queue_onehot = batch["queue_onehot"].float().to(device)
        labels = batch["instruction"].float().to(device)
        # forward pass
        outputs = model(grid, active_grid, queue_onehot.view(queue_onehot.size(0), -1))
        #backward pass
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss
    #calculate validation loss
    for batch in validation_dataloader:
        #Convert data to floats
        grid = batch["grid"].float().to(device)
        active_grid = batch["active_grid"].float().to(device)
        queue_onehot = batch["queue_onehot"].float().to(device)
        labels = batch["instruction"].float().to(device)
        # forward pass
        outputs = model(grid, active_grid, queue_onehot.view(queue_onehot.size(0), -1))
        #backward pass
        validation_loss = criterion(outputs, labels)
        running_val_loss += validation_loss
    train_loss = running_loss/len(dataloader)
    val_loss = running_val_loss/len(validation_dataloader)
    print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
    


cuda
Epoch 1, Loss: 1.1241, Validation Loss: 1.0206
Epoch 2, Loss: 0.9564, Validation Loss: 0.9651
Epoch 3, Loss: 0.9089, Validation Loss: 0.9281
Epoch 4, Loss: 0.8793, Validation Loss: 0.9142
Epoch 5, Loss: 0.8574, Validation Loss: 0.8952
Epoch 6, Loss: 0.8376, Validation Loss: 0.8759
Epoch 7, Loss: 0.8173, Validation Loss: 0.8754
Epoch 8, Loss: 0.8196, Validation Loss: 0.8577
Epoch 9, Loss: 0.7982, Validation Loss: 0.8536
Epoch 10, Loss: 0.7907, Validation Loss: 0.8458
Epoch 11, Loss: 0.7796, Validation Loss: 0.8392
Epoch 12, Loss: 0.7728, Validation Loss: 0.8515
Epoch 13, Loss: 0.7627, Validation Loss: 0.8468
Epoch 14, Loss: 0.7532, Validation Loss: 0.8430
Epoch 15, Loss: 0.7502, Validation Loss: 0.8422
Epoch 16, Loss: 0.7438, Validation Loss: 0.8370
Epoch 17, Loss: 0.7310, Validation Loss: 0.8418
Epoch 18, Loss: 0.7278, Validation Loss: 0.8445
Epoch 19, Loss: 0.7199, Validation Loss: 0.8333
Epoch 20, Loss: 0.7134, Validation Loss: 0.8522


In [12]:
#Export Model as TorchScript

# Create sample input tensors with the correct shape.
# Use a batch size of 1 for export.
sample_grid = torch.zeros(1, 10, 20)         # static grid
sample_active_grid = torch.zeros(1, 10, 20)    # active grid
sample_queue = torch.zeros(1, 64)            # one-hot encoded queue; shape (1, 8, 8)

# Trace the model using the sample inputs.
model.to(torch.device("cpu"))
traced_model = torch.jit.trace(model, (sample_grid, sample_active_grid, sample_queue))

# Save the traced model to a file
traced_model.save("tetris_net.pt")

C:\Users\20neu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\jit\_trace.py:1304: TracerWarning: Output nr 1. of the traced function does not match the corresponding output of the Python function. Detailed error:
Tensor-likes are not close!

Mismatched elements: 7 / 7 (100.0%)
Greatest absolute difference: 3.4097307324409485 at index (0, 1) (up to 1e-05 allowed)
Greatest relative difference: 17.100872982966326 at index (0, 1) (up to 1e-05 allowed)
  _check_trace(


In [13]:
print(torch.cuda.is_available())

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

True
cuda:0
